# FINN - End-to-End Flow (CNN CIFAR-10)
-----------------------------------------------------------------

Questo notebook sostituisce il modello **TFC-w1a1 (MNIST)** con una **CNN binarizzata W1A1 su CIFAR-10** usando Brevitas 0.10.0. La pipeline FINN (sezioni 2, 3, 4) è **identica** al notebook originale.

| | Originale | Questo notebook |
|---|---|---|
| Modello | TFC-w1a1 FC | CNN-w1a1 Conv |
| Dataset | MNIST 28×28×1 | CIFAR-10 32×32×3 |
| Brevitas | pretrained | training da zero |
| Pipeline FINN | invariata | invariata |

## Overview

The FINN compiler comes with many *transformations* that modify the ONNX representation of the network according to certain patterns. This notebook will demonstrate a *possible* sequence of such transformations to take a particular trained network all the way down to hardware, as shown in the figure below.

![](finn-design-flow-example.svg)

The white fields show the state of the network representation in the respective step. The colored fields represent the transformations that are applied to the network to achieve a certain result. The diagram is divided into 5 sections represented by a different color, each of it includes several flow steps. The flow starts in top left corner with Brevitas export (green section), followed by the preparation of the network (blue section) to bring the network into a form in which each layer can be represented by either a Vitis HLS function or a Verilog module. The model then gets passed to Vivado IPI stitching (orange section), and finally a PYNQ overlay bitfile is built and can be tested on a PYNQ board (yellow section).
There is an additional section for functional verification (red section) on the right side of the diagram, which we will not cover in this notebook. For details please take a look in the verification notebook which you can find [here](tfc_end2end_verification.ipynb)

This Jupyter notebook is organized based on the sections described above. We will use the following helper functions, `showSrc` to show source code of FINN library calls and `showInNetron` to show the ONNX model at the current transformation step. The Netron displays are interactive, but they only work when running the notebook actively and not on GitHub.

In [ ]:
from finn.util.visualization import showSrc, showInNetron
from finn.util.basic import make_build_dir
import os

build_dir = os.environ["FINN_BUILD_DIR"]

## Outline
-------------
1. [CNN CIFAR-10 — Training e Brevitas export](#brev_exp)
2. [Network preparation](#nw_prep)
3. [Hardware build](#vivado)
4. [PYNQ deployment](#hw_test)

## 1. CNN CIFAR-10 — Training e Brevitas export <a id='brev_exp'></a>

Definiamo e addestriamo una **CNN binarizzata W1A1 su CIFAR-10** con Brevitas 0.10.0.

### Architettura CNN-w1a1
```
Input (1, 3, 32, 32)
  ├── QuantIdentity (input quant UINT8)
  ├── QuantConv2d(3→64,  3×3) → BN → QuantHardTanh  W1A1  → MaxPool → 16×16×64
  ├── QuantConv2d(64→128, 3×3) → BN → QuantHardTanh W1A1  → MaxPool →  8×8×128
  ├── QuantConv2d(128→256,3×3) → BN → QuantHardTanh W1A1  → MaxPool →  4×4×256
  ├── Flatten → 4096
  ├── QuantLinear(4096→512) → BN → QuantHardTanh           W1A1
  └── QuantLinear(512→10)  → logits
```

> **QuantHardTanh con bit_width=1** è l'unica attivazione accettata da FINN: produce valori bipolari {-1,+1} con scala fissa 1.0.

### 1.1 Dipendenze

In [ ]:
import subprocess, sys, warnings
warnings.filterwarnings('ignore')

try:
    import torchvision
    print(f'torchvision: {torchvision.__version__}')
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'torchvision', '-q'])

import brevitas
print(f'brevitas: {brevitas.__version__}')

### 1.2 Dataset CIFAR-10

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt

# Augmentation aggressiva per massimizzare accuracy con sole 10 epoche
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010)),
])

DATA_DIR = '/tmp/cifar10_data'
os.makedirs(DATA_DIR, exist_ok=True)

trainset = torchvision.datasets.CIFAR10(root=DATA_DIR, train=True,  download=True, transform=transform_train)
testset  = torchvision.datasets.CIFAR10(root=DATA_DIR, train=False, download=True, transform=transform_test)

# num_workers=0 evita problemi di multiprocessing nel container FINN
trainloader = torch.utils.data.DataLoader(trainset, batch_size=256, shuffle=True,  num_workers=0, pin_memory=True)
testloader  = torch.utils.data.DataLoader(testset,  batch_size=256, shuffle=False, num_workers=0, pin_memory=True)

CLASS_NAMES = ['aereo','automobile','uccello','gatto','cervo',
               'cane','rana','cavallo','nave','camion']

print(f'Train: {len(trainset)} | Test: {len(testset)}')

# Visualizza campioni
imgs, labels = next(iter(trainloader))
fig, axes = plt.subplots(1, 8, figsize=(16, 2))
for i, ax in enumerate(axes):
    # de-normalize per visualizzazione
    img = imgs[i].numpy().transpose(1,2,0)
    img = img * np.array([0.2023,0.1994,0.2010]) + np.array([0.4914,0.4822,0.4465])
    ax.imshow(img.clip(0,1))
    ax.set_title(CLASS_NAMES[labels[i].item()], fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

### 1.3 Definizione CNN W1A1 — Brevitas 0.10.0

- `QuantHardTanh(bit_width=1)` → attivazione bipolare {-1,+1} richiesta da FINN
- `QuantConv2d / QuantLinear` con `weight_bit_width=1` → pesi binari
- `QuantIdentity(bit_width=8)` → quantizzazione input
- `BatchNorm` prima dell'attivazione → necessario per lo streamlining FINN

In [ ]:
import brevitas.nn as qnn
from brevitas.nn import QuantConv2d, QuantLinear, QuantHardTanh, QuantIdentity

print(f'Brevitas: {brevitas.__version__}')

# Helper functions — API Brevitas 0.10.0
def make_quant_conv(in_ch, out_ch, kernel, padding=1):
    return QuantConv2d(
        in_ch, out_ch, kernel,
        padding=padding, bias=False,
        weight_bit_width=1,
        return_quant_tensor=True
    )

def make_quant_linear(in_f, out_f, return_quant=True):
    return QuantLinear(
        in_f, out_f, bias=False,
        weight_bit_width=1,
        return_quant_tensor=return_quant
    )

def make_quant_act():
    # QuantHardTanh bit_width=1: produce {-1,+1} con scala=1.0
    # E' l'UNICA attivazione accettata da FINN per reti binarie
    return QuantHardTanh(
        bit_width=1,
        return_quant_tensor=True
    )


class CNNw1a1_CIFAR10(nn.Module):
    """
    CNN binarizzata W1A1 per CIFAR-10, compatibile con FINN.
    Pattern: QuantConv2d(W1) -> BN -> QuantHardTanh(A1) -> MaxPool
    """
    def __init__(self, num_classes=10):
        super().__init__()

        # Input quantization (8 bit)
        self.input_quant = QuantIdentity(
            bit_width=8,
            return_quant_tensor=True
        )

        # Blocco 1: 3 -> 64
        self.conv1  = make_quant_conv(3,   64, 3)
        self.bn1    = nn.BatchNorm2d(64)
        self.act1   = make_quant_act()
        self.pool1  = nn.MaxPool2d(2, 2)       # 16x16x64

        # Blocco 2: 64 -> 128
        self.conv2  = make_quant_conv(64,  128, 3)
        self.bn2    = nn.BatchNorm2d(128)
        self.act2   = make_quant_act()
        self.pool2  = nn.MaxPool2d(2, 2)       # 8x8x128

        # Blocco 3: 128 -> 256
        self.conv3  = make_quant_conv(128, 256, 3)
        self.bn3    = nn.BatchNorm2d(256)
        self.act3   = make_quant_act()
        self.pool3  = nn.MaxPool2d(2, 2)       # 4x4x256

        # FC
        self.fc1    = make_quant_linear(4*4*256, 512)
        self.bn_fc1 = nn.BatchNorm1d(512)
        self.act_fc1= make_quant_act()

        # Output (nessuna quantizzazione sull'uscita)
        self.fc2    = make_quant_linear(512, num_classes, return_quant=False)

    def _v(self, x):
        """Estrae il tensore da QuantTensor se necessario."""
        return x.value if hasattr(x, 'value') else x

    def forward(self, x):
        x = self.input_quant(x)

        x = self.conv1(x)
        x = self.bn1(self._v(x))
        x = self.act1(x)
        x = self.pool1(self._v(x))

        x = self.conv2(x)
        x = self.bn2(self._v(x))
        x = self.act2(x)
        x = self.pool2(self._v(x))

        x = self.conv3(x)
        x = self.bn3(self._v(x))
        x = self.act3(x)
        x = self.pool3(self._v(x))

        x = self._v(x).view(-1, 4*4*256)
        x = self.fc1(x)
        x = self.bn_fc1(self._v(x))
        x = self.act_fc1(x)
        x = self.fc2(self._v(x))
        return x


device = 'cuda' if torch.cuda.is_available() else 'cpu'
cnn = CNNw1a1_CIFAR10().to(device)
print(f'Dispositivo : {device}')
print(f'Parametri   : {sum(p.numel() for p in cnn.parameters()):,}')
print(cnn)

### 1.4 Training — 10 epoche ottimizzate

Strategie per massimizzare l'accuracy in sole 10 epoche:
- **Learning rate alto (5e-3)** con warmup + cosine decay aggressivo
- **Batch size 256** per sfruttare al massimo la CPU
- **Label smoothing 0.1** riduce overfitting precoce
- **Normalizzazione CIFAR-10 standard** (media/std reali del dataset)

In [ ]:
import torch.optim as optim
import time

EPOCHS = 10
CHECKPOINT_PATH = build_dir + '/cnn_cifar10_w1a1_best.pth'

# Label smoothing migliora la generalizzazione con poche epoche
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# LR alto per convergere velocemente in 10 epoche
optimizer = optim.Adam(cnn.parameters(), lr=5e-3, weight_decay=1e-4)

# OneCycleLR: warmup 3 epoche + decay rapido — ideale per training breve
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=5e-3,
    steps_per_epoch=len(trainloader),
    epochs=EPOCHS,
    pct_start=0.3,       # 30% warmup
    div_factor=10,
    final_div_factor=100
)

train_acc_history = []
test_acc_history  = []
best_acc = 0.0

def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            total   += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    return 100. * correct / total

print(f'Inizio training — {EPOCHS} epoche su {device}')
print(f'Batch size: 256 | LR max: 5e-3 | Label smoothing: 0.1')
print('-' * 65)

for epoch in range(EPOCHS):
    t0 = time.time()
    cnn.train()
    running_loss, correct, total = 0.0, 0, 0

    for inputs, labels in trainloader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = cnn(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        # Gradient clipping — stabilizza training reti binarie
        torch.nn.utils.clip_grad_norm_(cnn.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total   += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    train_acc = 100. * correct / total
    test_acc  = evaluate(cnn, testloader)
    elapsed   = time.time() - t0

    train_acc_history.append(train_acc)
    test_acc_history.append(test_acc)

    if test_acc > best_acc:
        best_acc = test_acc
        torch.save(cnn.state_dict(), CHECKPOINT_PATH)

    print(f'Epoch {epoch+1:2d}/{EPOCHS} | '
          f'Loss: {running_loss/len(trainloader):.3f} | '
          f'Train: {train_acc:.1f}% | '
          f'Test: {test_acc:.1f}% | '
          f'Best: {best_acc:.1f}% | '
          f'{elapsed:.0f}s')

print('-' * 65)
print(f'Training completato. Miglior accuracy: {best_acc:.2f}%')

In [ ]:
# Curve di training
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train_acc_history, marker='o', label='Train accuracy')
ax.plot(test_acc_history,  marker='s', label='Test accuracy')
ax.axhline(y=best_acc, color='r', linestyle='--', label=f'Best: {best_acc:.1f}%')
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy (%)')
ax.set_title('CNN W1A1 CIFAR-10 — 10 epoche')
ax.legend(); ax.grid(True)
plt.tight_layout(); plt.show()

# Carica best checkpoint
cnn.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
final_acc = evaluate(cnn, testloader)
print(f'Accuratezza finale (best checkpoint): {final_acc:.2f}%')

### 1.5 Brevitas export in QONNX

Step identico all'originale — solo il modello passato a `export_qonnx` è diverso.

In [ ]:
import onnx
from brevitas.export import export_qonnx
from qonnx.util.cleanup import cleanup as qonnx_cleanup

export_onnx_path = build_dir + '/cnn_cifar10_w1_a1.onnx'

# Input dummy CIFAR-10: batch=1, RGB, 32x32
dummy_input = torch.randn(1, 3, 32, 32).to(device)

cnn.eval()
export_qonnx(cnn, dummy_input, export_onnx_path)
qonnx_cleanup(export_onnx_path, out_file=export_onnx_path)

print(f'Modello QONNX: {export_onnx_path}')
print(f'Dimensione:    {os.path.getsize(export_onnx_path)/1024:.1f} KB')

In [ ]:
showInNetron(export_onnx_path)

In [ ]:
from qonnx.core.modelwrapper import ModelWrapper
from finn.transformation.qonnx.convert_qonnx_to_finn import ConvertQONNXtoFINN

model = ModelWrapper(export_onnx_path)
model = model.transform(ConvertQONNXtoFINN())

In [ ]:
model.save(build_dir + '/cnn_cifar10_w1_a1_finn.onnx')
showInNetron(build_dir + '/cnn_cifar10_w1_a1_finn.onnx')

## 2. Network preparation <a id='nw_prep'></a>

* [FINN-style Dataflow Architectures](#dataflow_arch)
* [Tidy-up transformations](#basic_trafo)
* [Streamlining](#streamline)
* [Conversion to HW layers](#hw_layers)
* [Creating a Dataflow Partition](#dataflow_partition)
* [Specialize layers](#specialize_layers)
* [Folding and Datawidth Converter, FIFO and TLastMarker Insertion](#folding)

### FINN-style Dataflow Architectures <a id='dataflow_arch'></a>

The key idea in FINN-style dataflow architectures is to parallelize across layers as well as within layers by dedicating a proportionate amount of compute resources to each layer, as illustrated in the figure below taken from the [FINN-R paper](https://arxiv.org/pdf/1809.04570.pdf):

![](finn-hw-arch.png)

### Tidy-up transformations <a id='basic_trafo'></a>

In [ ]:
from qonnx.transformation.general import GiveReadableTensorNames, GiveUniqueNodeNames, RemoveStaticGraphInputs
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.infer_datatypes import InferDataTypes
from qonnx.transformation.fold_constants import FoldConstants

model = model.transform(InferShapes())
model = model.transform(FoldConstants())
model = model.transform(GiveUniqueNodeNames())
model = model.transform(GiveReadableTensorNames())
model = model.transform(InferDataTypes())
model = model.transform(RemoveStaticGraphInputs())

model.save(build_dir + '/cnn_cifar10_w1_a1_tidy.onnx')

In [ ]:
showInNetron(build_dir + '/cnn_cifar10_w1_a1_tidy.onnx')

### Adding Pre- and Postprocessing <a id='prepost'></a>

In [ ]:
from finn.util.pytorch import ToTensor
from qonnx.transformation.merge_onnx_models import MergeONNXModels
from qonnx.core.datatype import DataType

model = ModelWrapper(build_dir + '/cnn_cifar10_w1_a1_tidy.onnx')
global_inp_name = model.graph.input[0].name
ishape = model.get_tensor_shape(global_inp_name)

totensor_pyt = ToTensor()
chkpt_preproc_name = build_dir + '/cnn_cifar10_w1_a1_preproc.onnx'
export_qonnx(totensor_pyt, torch.randn(ishape), chkpt_preproc_name)
qonnx_cleanup(chkpt_preproc_name, out_file=chkpt_preproc_name)
pre_model = ModelWrapper(chkpt_preproc_name)
pre_model = pre_model.transform(ConvertQONNXtoFINN())

model = model.transform(MergeONNXModels(pre_model))
global_inp_name = model.graph.input[0].name
model.set_tensor_datatype(global_inp_name, DataType['UINT8'])

model.save(build_dir + '/cnn_cifar10_w1_a1_with_preproc.onnx')
showInNetron(build_dir + '/cnn_cifar10_w1_a1_with_preproc.onnx')

In [ ]:
from qonnx.transformation.insert_topk import InsertTopK

model = model.transform(InsertTopK(k=1))
chkpt_name = build_dir + '/cnn_cifar10_w1_a1_pre_post.onnx'
model = model.transform(InferShapes())
model = model.transform(FoldConstants())
model = model.transform(GiveUniqueNodeNames())
model = model.transform(GiveReadableTensorNames())
model = model.transform(InferDataTypes())
model = model.transform(RemoveStaticGraphInputs())
model.save(chkpt_name)

showInNetron(build_dir + '/cnn_cifar10_w1_a1_pre_post.onnx')

### Streamlining <a id='streamline'></a>

In [ ]:
from finn.transformation.streamline import Streamline
showSrc(Streamline)

In [ ]:
from finn.transformation.streamline.reorder import MoveScalarLinearPastInvariants
import finn.transformation.streamline.absorb as absorb

model = ModelWrapper(build_dir + '/cnn_cifar10_w1_a1_pre_post.onnx')
model = model.transform(MoveScalarLinearPastInvariants())
model = model.transform(Streamline())
model.save(build_dir + '/cnn_cifar10_w1_a1_streamlined.onnx')
showInNetron(build_dir + '/cnn_cifar10_w1_a1_streamlined.onnx')

In [ ]:
from qonnx.transformation.bipolar_to_xnor import ConvertBipolarMatMulToXnorPopcount
from finn.transformation.streamline.round_thresholds import RoundAndClipThresholds
from qonnx.transformation.infer_data_layouts import InferDataLayouts
from qonnx.transformation.general import RemoveUnusedTensors

model = model.transform(ConvertBipolarMatMulToXnorPopcount())
model = model.transform(absorb.AbsorbAddIntoMultiThreshold())
model = model.transform(absorb.AbsorbMulIntoMultiThreshold())
model = model.transform(absorb.AbsorbScalarMulAddIntoTopK())
model = model.transform(RoundAndClipThresholds())
model = model.transform(InferDataLayouts())
model = model.transform(RemoveUnusedTensors())

model.save(build_dir + '/cnn_cifar10_w1a1_ready_for_hw_conversion.onnx')
showInNetron(build_dir + '/cnn_cifar10_w1a1_ready_for_hw_conversion.onnx')

### Conversion to HW layers <a id='hw_layers'></a>

In [ ]:
import finn.transformation.fpgadataflow.convert_to_hw_layers as to_hw
model = ModelWrapper(build_dir + '/cnn_cifar10_w1a1_ready_for_hw_conversion.onnx')
model = model.transform(to_hw.InferBinaryMatrixVectorActivation())
model = model.transform(to_hw.InferLabelSelectLayer())
model = model.transform(to_hw.InferThresholdingLayer())
model.save(build_dir + '/cnn_cifar10_w1_a1_hw_layers.onnx')
showInNetron(build_dir + '/cnn_cifar10_w1_a1_hw_layers.onnx')

### Creating a Dataflow Partition <a id='dataflow_partition'></a>

In [ ]:
from finn.transformation.fpgadataflow.create_dataflow_partition import CreateDataflowPartition

model = ModelWrapper(build_dir + '/cnn_cifar10_w1_a1_hw_layers.onnx')
parent_model = model.transform(CreateDataflowPartition())
parent_model.save(build_dir + '/cnn_cifar10_w1_a1_dataflow_parent.onnx')
showInNetron(build_dir + '/cnn_cifar10_w1_a1_dataflow_parent.onnx')

In [ ]:
from qonnx.custom_op.registry import getCustomOp
sdp_node = parent_model.get_nodes_by_op_type('StreamingDataflowPartition')[0]
sdp_node = getCustomOp(sdp_node)
dataflow_model_filename = sdp_node.get_nodeattr('model')
showInNetron(dataflow_model_filename)

In [ ]:
model = ModelWrapper(dataflow_model_filename)

### Specialize layers <a id='specialize_layers'></a>

In [ ]:
thresh_node = model.get_nodes_by_op_type('Thresholding')[0]
thresh_node_inst = getCustomOp(thresh_node)
thresh_node_inst.set_nodeattr('preferred_impl_style', 'hls')

In [ ]:
from finn.transformation.fpgadataflow.specialize_layers import SpecializeLayers

model = model.transform(SpecializeLayers('xc7z020clg400-1'))
model.save(build_dir + '/cnn_cifar10_w1_a1_specialize_layers.onnx')
showInNetron(build_dir + '/cnn_cifar10_w1_a1_specialize_layers.onnx')

### Folding <a id='folding'></a>

In [ ]:
model = ModelWrapper(build_dir + '/cnn_cifar10_w1_a1_specialize_layers.onnx')
fc_layers = model.get_nodes_by_op_type('MVAU_hls')

# (PE, SIMD, ram_style) per ogni layer
folding = [
    (16, 3,  'block'),   # conv1: SIMD=3 (RGB)
    (16, 64, 'block'),   # conv2
    (16, 64, 'block'),   # conv3
    (8,  64, 'block'),   # fc1
    (10, 8,  'auto'),    # fc2 output
]
for fcl, (pe, simd, ramstyle) in zip(fc_layers, folding):
    fcl_inst = getCustomOp(fcl)
    fcl_inst.set_nodeattr('PE', pe)
    fcl_inst.set_nodeattr('SIMD', simd)
    fcl_inst.set_nodeattr('ram_style', ramstyle)

thresh_layers = model.get_nodes_by_op_type('Thresholding_hls')
if thresh_layers:
    getCustomOp(thresh_layers[0]).set_nodeattr('PE', 3)

model.save(build_dir + '/cnn_cifar10_w1_a1_folded.onnx')
showInNetron(build_dir + '/cnn_cifar10_w1_a1_folded.onnx')

In [ ]:
from finn.transformation.fpgadataflow.insert_dwc import InsertDWC
from finn.transformation.fpgadataflow.insert_fifo import InsertFIFO
from finn.transformation.fpgadataflow.insert_tlastmarker import InsertTLastMarker
from finn.transformation.fpgadataflow.prepare_ip import PrepareIP
from finn.transformation.fpgadataflow.hlssynth_ip import HLSSynthIP
from finn.transformation.fpgadataflow.set_exec_mode import SetExecMode

model = ModelWrapper(build_dir + '/cnn_cifar10_w1_a1_folded.onnx')
model = model.transform(InsertDWC())
model = model.transform(InsertFIFO())
model = model.transform(InsertTLastMarker())
model = model.transform(GiveUniqueNodeNames())
model = model.transform(PrepareIP('xc7z020clg400-1', 10))
model = model.transform(HLSSynthIP())
model = model.transform(SetExecMode('rtlsim'))
model.save(build_dir + '/cnn_cifar10_w1_a1_ipgen.onnx')
showInNetron(build_dir + '/cnn_cifar10_w1_a1_ipgen.onnx')

## 3. Hardware build <a id='vivado'></a>

In [ ]:
from finn.transformation.fpgadataflow.create_stitched_ip import CreateStitchedIP

model = ModelWrapper(build_dir + '/cnn_cifar10_w1_a1_ipgen.onnx')
model = model.transform(CreateStitchedIP('xc7z020clg400-1', 10))
model.save(build_dir + '/cnn_cifar10_w1_a1_stitched_ip.onnx')
print('Stitched IP path: ' + model.get_metadata_prop('vivado_stitch_proj'))

In [ ]:
from finn.transformation.fpgadataflow.make_pynq_driver import MakePYNQDriver

model = model.transform(MakePYNQDriver('zynq-iodma'))
model.save(build_dir + '/cnn_cifar10_w1_a1_pynq_driver.onnx')

In [ ]:
from finn.transformation.fpgadataflow.make_zynq_proj import ZynqBuild

model = ModelWrapper(build_dir + '/cnn_cifar10_w1_a1_pynq_driver.onnx')
model = model.transform(ZynqBuild(platform='Pynq-Z1', period_ns=10))
model.save(build_dir + '/cnn_cifar10_w1_a1_post_synthesis.onnx')
print('Bitfile: ' + model.get_metadata_prop('bitfile'))

## 4. PYNQ deployment <a id='hw_test'></a>

In [ ]:
from shutil import copy, copytree

deployment_dir = make_build_dir(prefix='pynq_deployment_')

bitfile  = model.get_metadata_prop('bitfile')
hwh_file = model.get_metadata_prop('hw_handoff')
for dfile in [bitfile, hwh_file]:
    if dfile is not None:
        copy(dfile, deployment_dir)

pynq_driver_dir = model.get_metadata_prop('pynq_driver_dir')
copytree(pynq_driver_dir, deployment_dir, dirs_exist_ok=True)

In [ ]:
import numpy as np

raw_testset = torchvision.datasets.CIFAR10(
    root=DATA_DIR, train=False, download=False,
    transform=transforms.ToTensor()
)
x_sample, y_sample = raw_testset[0]
x_np = (x_sample.numpy() * 255).astype(np.uint8)   # (3, 32, 32)
x_np = np.expand_dims(x_np, axis=0)                # (1, 3, 32, 32)

plt.imshow(x_np[0].transpose(1,2,0))
plt.title(f'Classe: {CLASS_NAMES[y_sample]}')
plt.axis('off')
plt.show()

model = ModelWrapper(build_dir + '/cnn_cifar10_w1_a1_post_synthesis.onnx')
iname  = model.graph.input[0].name
ishape = model.get_tensor_shape(iname)
print('Expected network input shape is ' + str(ishape))
np.save(deployment_dir + '/input.npy', x_np.reshape(ishape))

In [ ]:
! ls {deployment_dir}

In [ ]:
from shutil import make_archive
make_archive('deploy-on-pynq-cnn-cifar10', 'zip', deployment_dir)

Download the zip (**File → Open**, checkbox → Download), copialo sulla board PYNQ e lancia:

```shell
unzip deploy-on-pynq-cnn-cifar10.zip -d finn-cnn-cifar10-demo
cd finn-cnn-cifar10-demo
sudo python3 -m pip install bitstring
sudo python3 driver.py --exec_mode=execute --batchsize=1 --bitfile=resizer.bit --inputfile=input.npy
```

### Validating the Accuracy on a PYNQ Board <a id='validation'></a>

```shell
sudo pip3 install git+https://github.com/fbcotter/dataset_loading.git@0.0.4#egg=dataset_loading
sudo python3 validate.py --dataset cifar10 --batchsize 1000
```

### Throughput Test on PYNQ Board <a id='throughput'></a>

```shell
sudo python3 driver.py --exec_mode=throughput_test --batchsize=1000 --bitfile=resizer.bit
```

The network metrics from the throughput test are saved in `nw_metrics.txt` on the PYNQ board.